# SAC arrival_v2 — history k=8 multi-seed (seed=0) on single_cross_s0 (1M, vanilla)

**Pre-context（commit `7be0956`）**：[`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md) §7.8 已闭环。`single_cross_s0` + vanilla SAC + history k=4→8 单变量 ablation 在 seed=42 上 **PASS**，闭合 §7.6.4 留下的 80pp catastrophic gap：

| §7.8 anchor (seed=42)         | final  | peak@step    | mean39 | OOB   | Gate |
|---                            |---:    |---           |---:    |---:   |---   |
| s0 k=4 vanilla (§7.6.4 base)  | 0.100  | 0.367 @ 975k | 0.221  | 0.667 | FAIL(3/5) |
| s0 k=4 + AsymCritic (§7.7)    | 0.167  | 0.267 @ 950k | 0.044  | 0.633 | FAIL(3/5) |
| s1 k=4 vanilla (§7.1 upper)   | 0.900  | 0.900 @ 725k | 0.497  | 0.100 | PASS |
| **s0 k=8 vanilla (§7.8)**     | **0.900** | **0.900 @ 475k** | **0.636** | **0.100** | **PASS（5/5）** |

**§7.8 PASS 的核心意义**：`single_cross_s0` 瓶颈不是 sensor 空间信息（s0 → s1），也不是 critic estimation accuracy（vanilla → AsymCritic），而是 **actor-side temporal information access**。给 actor 一个覆盖 20-40% 涡街周期（~4s vs 涡街 10-20s）的时序窗，单点 DVL 已足以反演主导脉动相位。

但 single-seed PASS 不足以支持 thesis-grade claim。**本 notebook 是 §7.8 multi-seed 巩固的 seed=0 run**：

> 完全 fix 其它所有 hyperparameter，把 §7.8 anchor 的 `--seed 42` 替换成 `--seed 0`，验证 k=8 PASS 是「方法-level 鲁棒效应」还是「seed-specific 偶然命中」。

**本 notebook 任务（pure vanilla + history k=8，单变量 seed swap）**：

| 维度                  | §7.8 anchor (seed=42) | 本 notebook (seed=0) |
|---                    |---                    |---                  |
| `--seed`              | 42                    | **0** ← 唯一变量    |
| `--history-length`    | 8                     | 8                   |
| algorithm             | vanilla SAC           | vanilla SAC         |
| sensor layout         | s0 (DVL-only)         | s0 (DVL-only)       |
| reward                | arrival_v2            | arrival_v2          |
| flow U / target       | 1.5 / 1.5             | 1.5 / 1.5           |
| total_steps           | 1M                    | 1M                  |
| num_envs              | 6                     | 6                   |
| benchmark             | `single_u15_cross_tgt15` | 同              |
| obs_dim               | 10×8 + 8 = 88         | 88                  |

**Seed 选择 = 0 的理由**：与 k=4 sister 套（§7.6.4 vanilla seed=42 + §7.7.1 vanilla seed=0 ✓ 已闭环）保持 seed 对齐，得到 **k=4 vs k=8 × seed=42 vs seed=0** 完整 2×2 paired contrast。

**Gate**（与 §7 / §7.6 / §7.7 / §7.8 同口径）：
- `final_success_rate ≥ 0.85`
- `last100k_mean ≥ 0.9 × peak`
- `final_oob_rate ≤ 0.10`
- `include_episode_context_obs == True`
- `timeout_bootstrap_semantics == 'terminal'`

**Paired verdict 规则**（驱动 §7.8 multi-seed update）：

| Verdict | 触发条件 | 解释 |
|---|---|---|
| **REPRO** | seed=0 也 PASS（5/5）且 \|Δfinal\| ≤ 0.15 且 \|Δmean\| ≤ 0.10 vs §7.8 anchor | k=8 effect 鲁棒，可写 §7.8 multi-seed claim |
| **HIGH-VARIANCE** | seed=0 PASS 但 \|Δfinal\| > 0.15 或 \|Δmean\| > 0.10 | 效应在但 noisy，需 seed=7 第三 anchor |
| **PARTIAL** | seed=0 STRONG-PARTIAL（final ∈ [0.5, 0.85)）| effect 存在但 seed 敏感，需 seed=7 加 budget |
| **REGRESS** | seed=0 final < 0.5 | k=8 effect 不鲁棒，§7.8 PASS 可能 seed=42 偶然；需第 3 seed 且改写 §7.8 narrative |

**输出根（与 §7.8 anchor 互不覆盖）**：
- `experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_0/`

**总预算**：~2.5h L4（1 Colab Pro+ session）。

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑（实时 stdout），与 §7.8 一致。


## 0. GPU sanity


In [1]:
!nvidia-smi | head -10


Mon May 18 14:14:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |


## 1. Mount Drive + cwd


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 2. Config — single phase（vanilla SAC + history k=8 + seed=0，与 §7.8 anchor 仅差一个 flag value）


In [3]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== SAC / env config (与 §7.8 anchor 严格一致，除 seed 外) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'
HISTORY_LENGTH = 8
TARGET_SPEED = 1.5
SEED = 0                                # ← 唯一与 §7.8 anchor (seed=42) 不同的值

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

# 显式拒绝所有 SAC 改进项 — 与 §7.8 anchor 一致
USE_ASYMMETRIC_CRITIC = False
USE_LAYERNORM = False
UPDATES_PER_STEP = 1
DROPOUT_RATE = 0.0

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

# Flow file（与 §7.1 / §7.6.4 / §7.7 / §7.8 严格一致）
SINGLE_FLOW = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'

# ==== Single phase — single_cross s0 k=8 seed=0 (X_*) ====
X_BENCHMARK_KEY = 'single_u15_cross_tgt15'
X_TASK_GEOMETRY = 'cross_stream'
X_FLOW_PATH = SINGLE_FLOW
X_TOTAL_STEPS = 1_000_000
X_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_0')
X_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_0')
X_MANIFEST_PATH = Path(f'benchmarks/{X_BENCHMARK_KEY}.json')

# Baselines（事后对比）
X_K8_ANCHOR_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42')      # §7.8 anchor
X_K4_S42_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')          # §7.6.4
X_K4_S0_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_0')             # §7.7.1 sister
X_S1_BASELINE_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')      # §7.1 upper

os.environ['PYTHONUNBUFFERED'] = '1'

print(f'PROBE_LAYOUT          = {PROBE_LAYOUT}')
print(f'OBJECTIVE             = {OBJECTIVE}')
print(f'HISTORY_LENGTH        = {HISTORY_LENGTH}')
print(f'SEED                  = {SEED}        ← 唯一与 §7.8 anchor (seed=42) 不同')
print(f'NUM_ENVS              = {NUM_ENVS}')
print(f'USE_ASYMMETRIC_CRITIC = {USE_ASYMMETRIC_CRITIC}')
print(f'USE_LAYERNORM         = {USE_LAYERNORM}')
print(f'UPDATES_PER_STEP      = {UPDATES_PER_STEP}')
print(f'DROPOUT_RATE          = {DROPOUT_RATE}')
print()
print(f'benchmark             : {X_BENCHMARK_KEY}')
print(f'geometry              : {X_TASK_GEOMETRY}')
print(f'total_steps           : {X_TOTAL_STEPS:,}')
print(f'expected obs_dim      : 10 * {HISTORY_LENGTH} + 8 (arrival_v2 context) = {10 * HISTORY_LENGTH + 8}')
print(f'run_root              : {X_RUN_ROOT}')
print(f'§7.8 k=8 anchor       : {X_K8_ANCHOR_ROOT}')
print(f'§7.6.4 k=4 seed=42    : {X_K4_S42_ROOT}')
print(f'§7.7.1 k=4 seed=0     : {X_K4_S0_ROOT}')
print(f'§7.1 s1 k=4 upper ref : {X_S1_BASELINE_ROOT}')


PROBE_LAYOUT          = s0
OBJECTIVE             = arrival_v2
HISTORY_LENGTH        = 8
SEED                  = 0        ← 唯一与 §7.8 anchor (seed=42) 不同
NUM_ENVS              = 6
USE_ASYMMETRIC_CRITIC = False
USE_LAYERNORM         = False
UPDATES_PER_STEP      = 1
DROPOUT_RATE          = 0.0

benchmark             : single_u15_cross_tgt15
geometry              : cross_stream
total_steps           : 1,000,000
expected obs_dim      : 10 * 8 + 8 (arrival_v2 context) = 88
run_root              : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_0
§7.8 k=8 anchor       : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42
§7.6.4 k=4 seed=42    : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42
§7.7.1 k=4 seed=0     : experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_0
§7.1 s1 k=4 upper ref : experiments/arrival_v2_prototype/singl

## 3. Preflight — flow / arrival_v2 candidate gate / reward unit tests / manifest / baseline 就位


In [4]:
# Flow file
fp = Path(X_FLOW_PATH)
if not fp.exists():
    raise FileNotFoundError(f'missing flow file: {fp}')
print(f'[OK] flow file: {fp}  ({fp.stat().st_size / 1e6:.1f} MB)')


[OK] flow file: wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy  (230.4 MB)


In [5]:
!python -u -m scripts.validate_arrival_v2_candidate



[undiscounted]
fast_success       144.473
slow_success       142.023
unsafe_success      91.048
timeout_near      -149.800
timeout_far       -229.800
late_oob          -274.500
fast_oob          -280.325
mid_oob           -319.450

[discounted_gamma_0.995]
fast_success        88.578
slow_success        54.683
unsafe_success      34.857
timeout_near       -39.480
timeout_far        -58.269
late_oob          -103.800
mid_oob           -190.006
fast_oob          -212.555

[discounted_shortcut] safe=69.330 risky=62.361

PASS: arrival_v2 candidate pre-integration gates passed.


In [6]:
!python -u -m pytest tests/test_reward_objective.py -q


...............                                                          [100%]
15 passed in 19.99s


In [7]:
if not X_MANIFEST_PATH.exists():
    !python -u -m scripts.generate_standard_benchmarks --benchmarks {X_BENCHMARK_KEY} --episodes {EVAL_EPISODES}
if not X_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest not generated: {X_MANIFEST_PATH}')
print(f'[OK] manifest ready: {X_MANIFEST_PATH}')


[OK] manifest ready: benchmarks/single_u15_cross_tgt15.json


In [8]:
# 检查 4 个对比 baseline 是否就位
for label, root, ref_final, ref_oob in [
    ('§7.8 k8 anchor s42  ', X_K8_ANCHOR_ROOT, 0.900, 0.100),
    ('§7.6.4 k4 vanilla s42', X_K4_S42_ROOT, 0.100, 0.667),
    ('§7.7.1 k4 vanilla s0 ', X_K4_S0_ROOT, 0.400, 0.200),
    ('§7.1 s1_k4 upper s42 ', X_S1_BASELINE_ROOT, 0.900, 0.100),
]:
    fp = root / 'results' / 'final_eval.json'
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        oob = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        match = '✓' if abs(f - ref_final) < 0.05 and abs(oob - ref_oob) < 0.05 else '✗ mismatch'
        print(f'[OK] {label}: final={f:.4f}  oob={oob:.4f}  counts={c}  {match}')
    else:
        print(f'[WARN] {label}: {fp} 不存在 (后续 §5 diff 会回退到 report 转载值)')


[OK] §7.8 k8 anchor s42  : final=0.9000  oob=0.1000  counts={'goal': 27, 'out_of_bounds': 3}  ✓
[OK] §7.6.4 k4 vanilla s42: final=0.1000  oob=0.6667  counts={'out_of_bounds': 20, 'timeout': 7, 'goal': 3}  ✓
[OK] §7.7.1 k4 vanilla s0 : final=0.4000  oob=0.2000  counts={'timeout': 12, 'out_of_bounds': 6, 'goal': 12}  ✓
[OK] §7.1 s1_k4 upper s42 : final=0.9000  oob=0.1000  counts={'goal': 27, 'out_of_bounds': 3}  ✓


## 4. Train — single_cross_s0 + history k=8 + seed=0 (1.0M, skip/resume)


In [ ]:
x_state_path = X_RUN_ROOT / 'trainer_state.json'
if x_state_path.exists():
    x_state = json.loads(x_state_path.read_text(encoding='utf-8'))
    x_current_step = int(x_state.get('env_step', 0))
else:
    x_current_step = 0
print(f'[state] X env_step = {x_current_step:,} / target {X_TOTAL_STEPS:,}')

if x_current_step >= X_TOTAL_STEPS:
    print(f'[skip] X already trained to {x_current_step:,} >= {X_TOTAL_STEPS:,}')
elif x_current_step > 0:
    print(f'[resume] X continuing from {x_current_step:,} -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(X_RUN_ROOT)} \
        --total-steps {X_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] X fresh start -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {X_FLOW_PATH} \
        --task-geometry {X_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {X_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(X_RUN_ROOT)} \
        --checkpoint-dir {str(X_CKPT_ROOT)}


[state] X env_step = 0 / target 1,000,000
[train] X fresh start -> 1,000,000
[train] episode=5 step=678 return=-310.07 success=False time=56.1s geometry=cross_stream history=8
[train] episode=10 step=1842 return=-455.98 success=False time=106.2s geometry=cross_stream history=8
[train] episode=15 step=2268 return=-404.30 success=False time=94.9s geometry=cross_stream history=8
[train] episode=20 step=2670 return=-324.46 success=False time=31.6s geometry=cross_stream history=8
[train] episode=25 step=3204 return=-285.89 success=False time=55.0s geometry=cross_stream history=8
[train] episode=30 step=3702 return=-416.50 success=False time=38.5s geometry=cross_stream history=8
[train] episode=35 step=4242 return=-432.68 success=False time=49.6s geometry=cross_stream history=8
[train] episode=40 step=5112 return=-497.95 success=False time=109.5s geometry=cross_stream history=8 | q1=787.244 actor=0.545 alpha=0.199
[train] episode=45 step=5556 return=-417.66 success=False time=90.6s geometry=

## 5. Summary + gate


In [ ]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'
    train_config_path = run_root / 'results' / 'train_config.txt'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    history_from_config = 'NA'
    if train_config_path.exists():
        for ln in train_config_path.read_text(encoding='utf-8').splitlines():
            if ln.strip().startswith('history_length='):
                history_from_config = ln.strip().split('=', 1)[1]
                break

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    mean_all = float(df['eval_success_rate'].mean()) if len(df) else 0.0
    n_evals_with_success = int((df['eval_success_rate'] > 0).sum()) if len(df) else 0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / s0 / k=8 / seed={SEED} / vanilla / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  full-traj mean        : {mean_all:.4f}   (39 evals)")
    print(f"  n_evals_with_success  : {n_evals_with_success} / {len(df)}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {trainer_state.get('observation_dim', 'NA')}   (expect 10*8+8=88)")
    print(f"  history_length        : {history_from_config}   (from train_config.txt)")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': int(history_from_config) if history_from_config != 'NA' else HISTORY_LENGTH,
        'seed': SEED,
        'total_steps': total_steps,
        'algorithm': 'sac_vanilla',
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'mean_success_full_trajectory': mean_all,
        'n_evals_with_success': n_evals_with_success,
        'n_evals_total': len(df),
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

x_summary = summarize_phase(
    X_RUN_ROOT,
    X_TOTAL_STEPS,
    'SINGLE_CROSS_S0_K8_SEED0',
    'single_cross_s0_k8_seed0_gate_summary.json',
)
X_PASS = x_summary['all_pass']
print()
print(f'X_PASS = {X_PASS}')


SINGLE_CROSS_S0_K8_SEED0  (arrival_v2 / s0 / k=8 / seed=0 / vanilla / 1,000,000 steps)
----------------------------------------------------------------------------------------------------
  final_success_rate    : 0.5000   gate >= 0.85
  peak_success_rate     : 0.5000   @ 925,002
  last100k_mean_success : 0.4750   gate >= 0.4500
  full-traj mean        : 0.2598   (39 evals)
  n_evals_with_success  : 32 / 39
  final_oob_rate        : 0.1333   gate <= 0.10
  obs_dim               : 96   (expect 10*8+8=88)
  history_length        : 8   (from train_config.txt)
  context_obs           : True
  timeout_bootstrap     : terminal
  termination           : {'timeout': 11, 'goal': 15, 'out_of_bounds': 4}

[last 16 eval rows]
 env_step  eval_success_rate  eval_return  eval_safety_cost  eval_time_s  eval_progress_ratio
   600000           0.366667   -90.768662         17.415726   136.143333             0.214618
   625002           0.466667   -25.784061         13.230684   136.523333             0.5

## 6. Paired verdict — k=8 seed=0 vs §7.8 anchor (seed=42) + 全表 6-run 对照


In [ ]:
def read_baseline(root: Path, ref_final: float, ref_oob: float, ref_mean: float = None):
    fp = root / 'results' / 'final_eval.json'
    log = root / 'results' / 'eval_log.csv'
    out = {'final': ref_final, 'oob': ref_oob, 'mean': ref_mean, 'peak': None, 'source': 'report'}
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        out['final'] = f
        out['oob'] = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        out['source'] = 'on-disk'
    if log.exists():
        dlog = pd.read_csv(log)
        out['mean'] = float(dlog['eval_success_rate'].mean())
        out['peak'] = float(dlog['eval_success_rate'].max())
    return out

print('=' * 100)
print('SINGLE_CROSS — k=8 multi-seed paired verdict (seed=0 vs §7.8 anchor seed=42)')
print('-' * 100)

# 本 run (k=8 seed=0)
k8s0_final = float(x_summary['final_success_rate'])
k8s0_oob   = float(x_summary['final_oob_rate'])
k8s0_peak  = float(x_summary['peak_success_rate'])
k8s0_mean  = float(x_summary.get('mean_success_full_trajectory', 0.0))
k8s0_nsucc = int(x_summary.get('n_evals_with_success', 0))
k8s0_ntot  = int(x_summary.get('n_evals_total', 0))

# 4 个 baseline（report-fallback 用 §7.8 / §7.6.4 / §7.7.1 / §7.1 转载值）
b_k8_s42 = read_baseline(X_K8_ANCHOR_ROOT,    0.900, 0.100, 0.636)   # §7.8 anchor
b_k4_s42 = read_baseline(X_K4_S42_ROOT,       0.100, 0.667, 0.221)   # §7.6.4
b_k4_s0  = read_baseline(X_K4_S0_ROOT,        0.400, 0.200, 0.218)   # §7.7.1 sister
b_s1     = read_baseline(X_S1_BASELINE_ROOT,  0.900, 0.100, 0.497)   # §7.1 upper

print()
print(f'{"config":<48}{"final":>10}{"mean":>10}{"oob":>10}{"peak":>10}{"source":>12}')
print('-' * 100)
print(f'{"vanilla s1_k4 s42 (§7.1 upper ref)":<48}{b_s1["final"]:>10.4f}{(b_s1["mean"] or float("nan")):>10.4f}{b_s1["oob"]:>10.4f}{(b_s1["peak"] or float("nan")):>10.4f}{b_s1["source"]:>12}')
print(f'{"vanilla s0_k4 s42 (§7.6.4 FAIL base)":<48}{b_k4_s42["final"]:>10.4f}{(b_k4_s42["mean"] or float("nan")):>10.4f}{b_k4_s42["oob"]:>10.4f}{(b_k4_s42["peak"] or float("nan")):>10.4f}{b_k4_s42["source"]:>12}')
print(f'{"vanilla s0_k4 s0  (§7.7.1 sister)":<48}{b_k4_s0["final"]:>10.4f}{(b_k4_s0["mean"] or float("nan")):>10.4f}{b_k4_s0["oob"]:>10.4f}{(b_k4_s0["peak"] or float("nan")):>10.4f}{b_k4_s0["source"]:>12}')
print(f'{"vanilla s0_k8 s42 (§7.8 anchor PASS)":<48}{b_k8_s42["final"]:>10.4f}{(b_k8_s42["mean"] or float("nan")):>10.4f}{b_k8_s42["oob"]:>10.4f}{(b_k8_s42["peak"] or float("nan")):>10.4f}{b_k8_s42["source"]:>12}')
print(f'{"vanilla s0_k8 s0  (THIS RUN)":<48}{k8s0_final:>10.4f}{k8s0_mean:>10.4f}{k8s0_oob:>10.4f}{k8s0_peak:>10.4f}{"on-disk":>12}')
print('=' * 100)

# Δ 行
delta_final = k8s0_final - b_k8_s42['final']
delta_mean  = k8s0_mean  - (b_k8_s42['mean'] or 0.0)
delta_oob   = k8s0_oob   - b_k8_s42['oob']

print()
print(f'{"contrast":<58}{"Δ final":>12}{"Δ mean":>12}{"Δ oob":>12}')
print('-' * 100)
print(f'{"k=8 seed=0 vs k=8 seed=42 (PAIRED)":<58}'
      f'{delta_final:>+12.4f}{delta_mean:>+12.4f}{delta_oob:>+12.4f}')
print(f'{"k=8 seed=0 vs k=4 seed=0 (cross-history paired)":<58}'
      f'{k8s0_final - b_k4_s0["final"]:>+12.4f}'
      f'{(k8s0_mean - (b_k4_s0["mean"] or 0)):>+12.4f}'
      f'{k8s0_oob - b_k4_s0["oob"]:>+12.4f}')
print(f'{"k=8 seed=0 vs s1_k4 seed=42 (gap-to-upper)":<58}'
      f'{k8s0_final - b_s1["final"]:>+12.4f}'
      f'{(k8s0_mean - (b_s1["mean"] or 0)):>+12.4f}'
      f'{k8s0_oob - b_s1["oob"]:>+12.4f}')
print('=' * 100)
print()
print(f'k=8 seed=0 evals_with_success: {k8s0_nsucc} / {k8s0_ntot}  (k=8 seed=42 was 37/39, k=4 seed=42 was 35/39, k=4 seed=0 was 34/39)')

# Paired verdict — 4 档
THRESH_FINAL = 0.15   # paired |Δfinal| 容忍度
THRESH_MEAN  = 0.10   # paired |Δmean| 容忍度
abs_delta_final = abs(delta_final)
abs_delta_mean  = abs(delta_mean)
seed0_pass = (k8s0_final >= PASS_FINAL_SUCCESS) and (k8s0_oob <= PASS_OOB_RATE)
seed0_strong_partial = (k8s0_final >= 0.50) and (k8s0_final < PASS_FINAL_SUCCESS)
seed0_regress = (k8s0_final < 0.50)

if seed0_pass and abs_delta_final <= THRESH_FINAL and abs_delta_mean <= THRESH_MEAN:
    verdict = (
        'REPRO — k=8 effect 跨 seed 鲁棒（两 seed 都 PASS 且 |Δfinal|≤0.15 |Δmean|≤0.10）；'
        '可写 §7.8 multi-seed claim；3rd seed (=7) 仅为 thesis-grade triplication'
    )
elif seed0_pass and (abs_delta_final > THRESH_FINAL or abs_delta_mean > THRESH_MEAN):
    verdict = (
        'HIGH-VARIANCE — 两 seed 都 PASS 但 Δ 超阈值；effect 在但 noisy；'
        '需 seed=7 第三 anchor 才能写 §7.8 multi-seed；现阶段 §7.8 claim 为 "seed=42+0 PASS, var TBD"'
    )
elif seed0_strong_partial:
    verdict = (
        'PARTIAL — seed=0 STRONG-PARTIAL（final ∈ [0.5, 0.85)）；'
        'effect 存在但 seed 敏感；需 seed=7 + 可能加 budget 或 init scheme 看是否能收敛；'
        '§7.8 claim 暂时 reframe 为 "k=8 部分缓解 information bottleneck"'
    )
else:  # seed0_regress
    verdict = (
        'REGRESS — seed=0 final<0.50；强证据 §7.8 PASS 可能 seed=42 偶然命中；'
        '必须跑 seed=7 + 重新 audit §7.8 narrative；可能需要回退到 "k=4→8 effect seed-dependent" 的 honest framing'
    )

print()
print(f'>>> verdict: {verdict}')

# 落盘 ablation summary
ablation_out = {
    'experiment': 'arrival_v2_s0_cross_k8_seed0_replication',
    'seed': SEED,
    'benchmark': X_BENCHMARK_KEY,
    'probe_layout': PROBE_LAYOUT,
    'history_length': HISTORY_LENGTH,
    'total_steps': X_TOTAL_STEPS,
    'algorithm': 'sac_vanilla',
    'cli_diff_vs_§7.8_anchor': '--seed 42 → --seed 0',
    'results': {
        'k8_s0_thisrun': {
            'final': k8s0_final, 'mean': k8s0_mean, 'oob': k8s0_oob, 'peak': k8s0_peak,
            'n_evals_with_success': k8s0_nsucc, 'n_evals_total': k8s0_ntot,
        },
        'k8_s42_anchor_§7.8': b_k8_s42,
        'k4_s42_§7.6.4': b_k4_s42,
        'k4_s0_§7.7.1_sister': b_k4_s0,
        's1_k4_s42_§7.1_upper': b_s1,
    },
    'delta_vs_§7.8_seed42_paired': {
        'final_pp': round(delta_final * 100, 2),
        'mean_pp':  round(delta_mean  * 100, 2),
        'oob_pp':   round(delta_oob   * 100, 2),
    },
    'all_pass': bool(x_summary['all_pass']),
    'verdict': verdict,
}
out_dir = Path('experiments/arrival_v2_prototype/s0_cross_k8_seed0_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(ablation_out, indent=2), encoding='utf-8')
print(f'\n[saved] {out_path}')


SINGLE_CROSS — k=8 multi-seed paired verdict (seed=0 vs §7.8 anchor seed=42)
----------------------------------------------------------------------------------------------------

config                                               final      mean       oob      peak      source
----------------------------------------------------------------------------------------------------
vanilla s1_k4 s42 (§7.1 upper ref)                  0.9000    0.4966    0.1000    0.9000     on-disk
vanilla s0_k4 s42 (§7.6.4 FAIL base)                0.1000    0.2205    0.6667    0.3667     on-disk
vanilla s0_k4 s0  (§7.7.1 sister)                   0.4000    0.2179    0.2000    0.5333     on-disk
vanilla s0_k8 s42 (§7.8 anchor PASS)                0.9000    0.6359    0.1000    0.9000     on-disk
vanilla s0_k8 s0  (THIS RUN)                        0.5000    0.2598    0.1333    0.5000     on-disk

contrast                                                       Δ final      Δ mean       Δ oob
------------------